In [3]:
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import matplotlib.font_manager as fm
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats

In [7]:
CCTV_FILE      = "./DATA/서울시 자치구 CCTV 설치현황.xlsx"
POP_FILE       = "./DATA/등록인구_20260512150716.xlsx"
CRIME_CSV      = "./DATA/5대+범죄+발생현황_20260512140024.csv"
LOC_FILE       = "./DATA/5대+범죄+발생장소별+현황.xlsx"
CRIME_NEW_FILE = "./DATA/crime(2015-2024).xlsx"
UTIL_FILE      = "./DATA/util_clean.xlsx"
ADULT_FILE     = "./DATA/서울시 유흥주점영업 인허가 정보.xlsx"
CCTV_FILE = r"C:\Users\Win11Pro\Documents\과제\cctv\DATA\경찰청_범죄 발생 지역별 통계_20241231.xlsx"

In [8]:
raw = pd.read_excel(CCTV_FILE)
raw.shape

(38, 249)

In [21]:
def load_cctv():
    raw = pd.read_excel(CCTV_FILE, header=2)
    raw = raw.drop(columns=["Unnamed: 0"], errors="ignore")
    raw = raw.iloc[:-2]
    raw = raw[raw["구분"] != "계"].copy()
    year_cols = [c for c in raw.columns if str(c).endswith("년")]
    raw[year_cols] = raw[year_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    return raw.set_index("순번")

def load_crime_csv():
    return pd.read_csv(CRIME_CSV, encoding="utf-8")

def load_population():
    return pd.read_excel(POP_FILE)

def load_crime_loc():
    return pd.read_excel(LOC_FILE)

def load_cctv_new():
    raw   = pd.read_excel(CCTV_FILE, header=2)
    total = raw[raw["구분"] == "계"]
    years = [f"{y}년" for y in range(2015, 2025)]
    vals  = total[years].iloc[0].values
    return pd.DataFrame({"연도": [str(y) for y in range(2015, 2025)],
                         "CCTV누적": vals.astype(float)})

def load_util():
    return pd.read_excel(UTIL_FILE)

def load_adult():
    raw   = pd.read_excel(ADULT_FILE)
    adult = raw[raw['영업상태명'] == '영업/정상'].copy()
    def extract_gu(row):
        addr = row['도로명주소'] if pd.notna(row['도로명주소']) else row['지번주소']
        if pd.isna(addr):
            return None
        parts = str(addr).split()
        return parts[1] if len(parts) >= 2 else None
    adult['지역구'] = adult.apply(extract_gu, axis=1)
    gu_counts = adult['지역구'].value_counts().reset_index()
    gu_counts.columns = ['자치구', '유흥업소수']
    return gu_counts

def build_crime_yearly():
    headers    = pd.read_excel(CRIME_NEW_FILE, header=None, nrows=4)
    data       = pd.read_excel(CRIME_NEW_FILE, skiprows=4, header=None)
    crime_list = ["절도","폭력","살인","강도","강간·강제추행"]
    rows = []
    for col in range(2, len(headers.columns)):
        try:    yr = str(int(float(headers.iloc[0, col])))
        except: yr = str(headers.iloc[0, col]).strip()
        c_type = str(headers.iloc[2, col]).strip()
        s_type = str(headers.iloc[3, col]).strip()
        if yr.isdigit() and 2015 <= int(yr) <= 2024 and s_type == "발생" and c_type in crime_list:
            val = pd.to_numeric(data.loc[data[1] == "소계", col], errors="coerce").fillna(0).sum()
            rows.append({"연도": yr, "유형": c_type, "건수": int(val)})
    return pd.DataFrame(rows)

def build_df_final():
    cctv      = load_cctv()
    crime_raw = load_crime_csv()
    pop_raw   = load_population()

    pop_24 = pop_raw[pop_raw["동별(1)"].isna() & pop_raw["동별(2)"].notna()][
        ["동별(2)", "2024.1"]].copy()
        # ["동별(2)", "2024 4/4.1"]].copy()
    pop_24.columns = ["자치구", "인구수"]
    pop_24["자치구"] = pop_24["자치구"].astype(str).str.strip()
    pop_24["인구수"] = pd.to_numeric(pop_24["인구수"].astype(str).str.replace(",", ""), errors="coerce")
    pop_24 = pop_24.dropna(subset=["인구수"])
    pop_24["인구수"] = pop_24["인구수"].astype(int)

    cctv_24 = cctv[["구분", "2024년"]].rename(columns={"구분": "자치구", "2024년": "CCTV수"}).copy()
    cctv_24["CCTV수"] = pd.to_numeric(cctv_24["CCTV수"], errors="coerce")

    crime_filt = crime_raw[[c for c in crime_raw.columns if "." not in str(c)]].copy()
    crime_filt = crime_filt.drop(0).rename(columns={"자치구별(2)": "자치구"})
    if "자치구별(1)" in crime_filt.columns:
        crime_filt = crime_filt.drop(columns=["자치구별(1)"])
    crime_24 = crime_filt[["자치구", "2024"]].rename(columns={"2024": "발생건수"}).copy()
    crime_24["발생건수"] = pd.to_numeric(crime_24["발생건수"], errors="coerce")

    df = pd.merge(pop_24, cctv_24, on="자치구")
    df = pd.merge(df, crime_24, on="자치구")
    df = df.dropna()
    df["범죄율"]        = (df["발생건수"] / df["인구수"]) * 100_000
    df["CCTV_밀도"]     = (df["CCTV수"]   / df["인구수"]) * 10_000
    df["CCTV당_범죄수"] = df["발생건수"]  / df["CCTV수"]
    df.index = range(1, len(df) + 1)
    return df

def build_crime_trend():
    cctv      = load_cctv()
    crime_raw = load_crime_csv()
    pop_raw   = load_population()
    CRIME_TYPES   = ["살인","강도","강간·강제추행","절도","폭력"]
    CRIME_OFFSETS = [2, 4, 6, 8, 10]
    rows = []
    for year in range(2015, 2025):
        yr = str(year)
        try:
            pop_total = pd.to_numeric(
                str(pop_raw.loc[pop_raw["동별(2)"] == "소계", f"{yr}.1"].values[0]).replace(",", ""),
                errors="coerce")
        except Exception:
            continue
        cctv_col = f"{yr}년"
        cctv_sum = cctv[cctv_col].sum() if cctv_col in cctv.columns else np.nan
        row_sum  = crime_raw[(crime_raw["자치구별(1)"] == "합계") & (crime_raw["자치구별(2)"] == "소계")]
        if row_sum.empty:
            row_sum = crime_raw[crime_raw["자치구별(1)"] == "합계"].iloc[[0]]
        total_crime      = pd.to_numeric(str(row_sum[yr].values[0]).replace(",", ""), errors="coerce")
        crime_rate_total = (total_crime / pop_total) * 100_000
        type_rates = []
        for offset in CRIME_OFFSETS:
            col = f"{yr}.{offset}"
            val = pd.to_numeric(str(row_sum[col].values[0]).replace(",", ""), errors="coerce")
            type_rates.append((val / pop_total) * 100_000)
        rows.append([yr, cctv_sum, pop_total, total_crime, crime_rate_total] + type_rates)
    df = pd.DataFrame(rows, columns=["연도","CCTV수량","인구수","총범죄수","총범죄율"] + CRIME_TYPES)
    df["연도"] = df["연도"].astype(str)


In [22]:
load_cctv()

,구분,2015년,2016년,2017년,2018년,2019년,2020년,2021년,2022년,2023년,2024년,2025년
순번,,,,,,,,,,,,
1,종로구,935.0,1066.0,1225.0,1322.0,1327.0,1510.0,1573.0,1812.0,1872.0,2154.0,2885.0
2,중구,363.0,565.0,838.0,1174.0,1242.0,1482.0,1911.0,2026.0,2157.0,2567.0,2861.0
3,용산구,1398.0,1689.0,1831.0,1888.0,1915.0,2058.0,2321.0,2531.0,2897.0,3202.0,3357.0
4,성동구,1089.0,1328.0,2103.0,2390.0,2833.0,3162.0,3519.0,3627.0,3871.0,4084.0,4226.0
5,광진구,638.0,657.0,1112.0,1586.0,2308.0,2481.0,3111.0,3370.0,3421.0,4370.0,4590.0
6,동대문구,1202.0,1425.0,1535.0,1775.0,2061.0,2166.0,2471.0,2592.0,3077.0,3602.0,4001.0
7,중랑구,751.0,898.0,1047.0,1203.0,2250.0,3165.0,3592.0,3856.0,4163.0,5009.0,5161.0
8,성북구,1035.0,1534.0,1940.0,2542.0,3238.0,3440.0,3815.0,4014.0,4216.0,4479.0,4994.0
9,강북구,608.0,840.0,841.0,1159.0,1656.0,2337.0,2960.0,3184.0,3191.0,3430.0,4434.0


In [23]:
load_cctv_new()

,연도,CCTV누적
0,2015,26321.0
1,2016,33013.0
2,2017,40512.0
3,2018,49222.0
4,2019,58139.0
5,2020,67281.0
6,2021,74408.0
7,2022,80005.0
8,2023,86810.0
9,2024,99758.0


In [24]:
load_adult()


,자치구,유흥업소수
0,중구,206
1,관악구,190
2,종로구,181
3,강남구,165
4,영등포구,131
5,강서구,126
6,강동구,96
7,송파구,85
8,은평구,75
9,강북구,69


In [25]:
build_crime_yearly()

,연도,유형,건수
0,2015,살인,163
1,2015,강도,276
2,2015,강간·강제추행,5449
3,2015,절도,55307
4,2015,폭력,65206
5,2016,살인,148
6,2016,강도,262
7,2016,강간·강제추행,6002
8,2016,절도,46857
9,2016,폭력,64570


In [26]:
build_df_final()

c:\Users\Win11Pro\anaconda3\envs\PY_10\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,자치구,인구수,CCTV수,발생건수,범죄율,CCTV_밀도,CCTV당_범죄수
1,종로구,149608,2154.0,2765.0,1848.163200,143.976258,1.283658
2,중구,131214,2567.0,2955.0,2252.046276,195.634612,1.151149
3,용산구,217194,3202.0,3322.0,1529.508182,147.425804,1.037477
4,성동구,281289,4084.0,2117.0,752.606750,145.188756,0.518364
5,광진구,348652,4370.0,2870.0,823.170382,125.339880,0.656751
6,동대문구,358603,3602.0,3216.0,896.813468,100.445339,0.892837
7,중랑구,385349,5009.0,3169.0,822.371409,129.986065,0.632661
8,성북구,435037,4479.0,2184.0,502.026264,102.956760,0.487609
9,강북구,289374,3430.0,2289.0,791.017852,118.531727,0.667347
10,도봉구,306032,2623.0,1741.0,568.894756,85.709991,0.663744


In [27]:
build_crime_trend()

c:\Users\Win11Pro\anaconda3\envs\PY_10\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
